# CMT316 - VOC Localisation (Faster R-CNN Pipeline)

This notebook follows an end-to-end Computer Vision Machine Learning pipeline for VOC object localisation using Faster R-CNN.

## Pipeline Overview
1. Requirements and runtime setup.
2. Dataset bootstrap and split integrity checks.
3. Shared VOC parsing + EDA diagnostics.
4. Model setup, training modes, and experiment execution.
5. Validation-driven model selection.
6. Export predictions and report-ready evaluation outputs.

## Important evaluation note
- Full multi-class VOC evaluation is performed on the authors' official evaluation server for final submission.
- Inside this notebook, detailed local evaluation and visualisation focus on the `person` class only (as required for local reporting).

## Table of Contents
- [1. Environment and Requirements](#1-environment-and-requirements)
- [2. Dataset Organisation and Bootstrap](#2-dataset-organisation-and-bootstrap)
- [3. Shared VOC Data Construction](#3-shared-voc-data-construction)
- [4. Exploratory Data Analysis](#4-exploratory-data-analysis)
- [5. Faster R-CNN Setup and Training Utilities](#5-faster-r-cnn-setup-and-training-utilities)
- [6. Experiment Suite (Three Explicit Runs)](#6-experiment-suite-five-explicit-runs)
- [7. Evaluation and Visualisation (All Classes)](#7-evaluation-and-visualisation-person-class)
- [8. Predict and Export](#8-predict-and-export)

In [ ]:
from pathlib import Path
import re
import shutil
import zipfile

import gdown

DATASET_LINK = 'https://drive.google.com/file/d/1M8EPREjTLAqY3VAjfruDcxtaudXy4xm2/view?usp=share_link'

PROJECT_ROOT = Path.cwd()
ARCHIVE_DIR = PROJECT_ROOT / 'archives'
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

FORCE_REDOWNLOAD = False
FORCE_REEXTRACT = False


def extract_drive_file_id(link: str) -> str:
    patterns = [r'/file/d/([a-zA-Z0-9_-]+)', r'id=([a-zA-Z0-9_-]+)']
    for pattern in patterns:
        match = re.search(pattern, link)
        if match:
            return match.group(1)
    raise ValueError(f'Could not parse Google Drive file id from: {link}')



def find_dataset_root(root: Path) -> Path | None:
    # The extracted archive can have variable nesting; find the actual dataset folder.
    for path in root.rglob('*'): # Look for any directory that contains JPEGImages or Annotations
        if path.is_dir() and (path / 'JPEGImages').exists() or (path / 'Annotations').exists():
            return path
    return None



def ensure_archive(link: str, output_zip: Path, force_download: bool = False) -> Path:
    if output_zip.exists() and not force_download:
        print(f'Archive already exists: {output_zip.name}')
        return output_zip
    if output_zip.exists() and force_download:
        output_zip.unlink()

    file_id = extract_drive_file_id(link)
    direct_url = f'https://drive.google.com/uc?id={file_id}'
    print(f'Downloading {output_zip.name}...')
    gdown.download(url=direct_url, output=str(output_zip), quiet=False)

    if not output_zip.exists() or output_zip.stat().st_size == 0:
        raise RuntimeError(f'Failed to download archive: {output_zip.name}')
    return output_zip



def install_voc_split(split_name: str, archive_path: Path, data_root: Path, force_extract: bool = False) -> Path:
    target = data_root / split_name
    if target.exists() and any(target.iterdir()) and not force_extract:
        print(f'Dataset already extracted: {target}')
        return target

    tmp_dir = data_root / '_tmp_extract' / split_name
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    tmp_dir.mkdir(parents=True, exist_ok=True)

    print(f'Extracting {archive_path.name} to {tmp_dir}...')
    with zipfile.ZipFile(archive_path, 'r') as zf:
        zf.extractall(tmp_dir)

    dataset_dir = find_dataset_root(tmp_dir)
    if dataset_dir is None:
        if target.parent.exists() and force_extract:
            shutil.rmtree(target.parent)
        target.parent.mkdir(parents=True, exist_ok=True)

        if target.exists():
            shutil.rmtree(target)
        shutil.move(str(tmp_dir), str(target))
        print(f'Dataset extracted to: {target}')
        return target

    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(dataset_dir, target)
    shutil.rmtree(tmp_dir)
    print(f'Dataset extracted to: {target}')
    return target


dataset_zip = ensure_archive(DATASET_LINK, ARCHIVE_DIR / 'dataset.zip', force_download=FORCE_REDOWNLOAD)
voc_dataset_root = install_voc_split('voc_dataset', dataset_zip, PROJECT_ROOT, force_extract=FORCE_REEXTRACT)

print('Project Root:', PROJECT_ROOT)
print('Dataset ready at:', voc_dataset_root)


## 2. Dataset Organisation and Bootstrap

This notebook uses the already materialized versioned VOC split stored under `version_2/data/`:

- `version_2/data/clean_ids/{train,val,dev_test,test}.txt`
- `version_2/data/images/{train,val,dev_test,test}/`
- `version_2/data/annotations/{train,val,dev_test,test}/`

Run this section to verify the local layout before training. No download or extraction is performed here.

In [ ]:
from pathlib import Path

requirements_path = Path('requirements.txt')

if requirements_path.exists():
    %pip install torchmetrics -r requirements.txt
else:
    %pip install torchmetrics numpy pandas matplotlib seaborn pillow torch torchvision transformers accelerate ultralytics torchmetrics

### Legacy bootstrap note

The archive-based bootstrap used in earlier drafts has been removed. The notebook now uses the versioned split layout under `version_2/data/` throughout.

In [ ]:
from pathlib import Path

# Verify the layout of the extracted or existing dataset
SPLIT_DIR = voc_dataset_root / 'version_2' / 'data' / 'clean_ids'
IMAGE_ROOT = voc_dataset_root / 'version_2' / 'data' / 'images'
ANNOTATION_ROOT = voc_dataset_root / 'version_2' / 'data' / 'annotations'
EXPECTED_SPLITS = ['train', 'val', 'dev_test', 'test']

missing_paths = [path for path in [SPLIT_DIR, IMAGE_ROOT, ANNOTATION_ROOT] if not path.exists()]
if missing_paths:
    print(f'Warning: Some expected paths are missing in {voc_dataset_root}')
else:
    print('✅ Dataset layout verified.')

print('Bootstrap Dataset root:', voc_dataset_root)
if SPLIT_DIR.exists():
    print('Split files:', sorted(path.name for path in SPLIT_DIR.glob('*.txt')))


## 3. Shared VOC Data Construction

This stage initialises the runtime, dataset paths, class dictionaries, and split-aware parsing utilities.

The implementation prioritises split hygiene:
- train/val leakage is prevented
- test remains untouched for final reporting
- IDs are robustly parsed from both one-column and two-column VOC list formats

In [ ]:
from pathlib import Path
import random
import xml.etree.ElementTree as ET

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from torchvision.transforms import v2 as T
from torchvision import tv_tensors
from PIL import Image
from torchmetrics.detection import MeanAveragePrecision
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)
print('Seed:', SEED)

In [ ]:
PROJECT_ROOT = Path.cwd()
DATA_ROOT_CANDIDATES = [
    PROJECT_ROOT / 'voc_dataset' / 'version_2' / 'data',
    PROJECT_ROOT / 'version_2' / 'data',
]
for candidate in DATA_ROOT_CANDIDATES:
    if candidate.exists():
        DATA_ROOT = candidate
        break
else:
    DATA_ROOT = DATA_ROOT_CANDIDATES[0]

SPLIT_ID_DIR = DATA_ROOT / 'clean_ids'
IMAGE_ROOT = DATA_ROOT / 'images'
ANNOTATION_ROOT = DATA_ROOT / 'annotations'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
TRAIN_IMAGE_DIR = IMAGE_ROOT / 'train'
VAL_IMAGE_DIR = IMAGE_ROOT / 'val'
DEV_TEST_IMAGE_DIR = IMAGE_ROOT / 'dev_test'
TEST_IMAGE_DIR = IMAGE_ROOT / 'test'
TRAIN_ANN_DIR = ANNOTATION_ROOT / 'train'
VAL_ANN_DIR = ANNOTATION_ROOT / 'val'
DEV_TEST_ANN_DIR = ANNOTATION_ROOT / 'dev_test'
TEST_ANN_DIR = ANNOTATION_ROOT / 'test'
PREDICTIONS_DIR = ARTIFACTS_DIR / 'predictions'
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_RUNS_DIR = ARTIFACTS_DIR / 'experiment_runs'
EXPERIMENT_RUNS_DIR.mkdir(parents=True, exist_ok=True)

VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
CLASS_TO_ID = {name: idx for idx, name in enumerate(VOC_CLASSES)}
ID_TO_CLASS = {idx: name for name, idx in CLASS_TO_ID.items()}

PREDICTION_COLUMNS = [
    'image_id', 'split', 'source_model', 'class_id', 'cls',
    'xmin', 'ymin', 'xmax', 'ymax', 'score'
]

for path in [TRAIN_IMAGE_DIR, VAL_IMAGE_DIR, DEV_TEST_IMAGE_DIR, TEST_IMAGE_DIR, TRAIN_ANN_DIR, VAL_ANN_DIR, DEV_TEST_ANN_DIR, TEST_ANN_DIR]:
    if not path.exists():
        print(f'Warning: Missing expected VOC folder: {path}')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('SPLIT_ID_DIR:', SPLIT_ID_DIR)


### 3.1 Split Parsing and Annotation Tables

This section builds image-level and object-level tables from VOC XML files and enforces clean split boundaries.

Outputs produced here are reused by both EDA and model training.

In [ ]:
def read_ids_from_file(path: Path) -> list[str]:
    if not path.exists():
        return []

    # Handles both formats:
    # 1) "image_id"
    # 2) "image_id <object_index>" (VOC Layout lists)
    ids = []
    seen = set()
    for line in path.read_text().splitlines():
        tokens = line.strip().split()
        if not tokens:
            continue
        image_id = tokens[0]
        if image_id not in seen:
            ids.append(image_id)
            seen.add(image_id)
    return ids



def load_split_ids(split_name: str) -> list[str]:
    # Split IDs are materialized under version_2/data/clean_ids.
    split_path = SPLIT_ID_DIR / f'{split_name}.txt'
    split_ids = read_ids_from_file(split_path)
    if not split_ids:
        raise FileNotFoundError(f'Missing split file: {split_path}')
    return split_ids



# Original parse_annotation replaced
def parse_annotation(xml_path: Path, image_id: str, split_name: str) -> tuple[dict, list[dict]]:
    root = ET.parse(xml_path).getroot()

    size_node = root.find('size')
    width = int(size_node.findtext('width', default='0')) if size_node is not None else 0
    height = int(size_node.findtext('height', default='0')) if size_node is not None else 0

    records = []
    for obj in root.findall('object'):
        cls = obj.findtext('name', default='').strip()
        if cls not in CLASS_TO_ID:
            continue

        box = obj.find('bndbox')
        if box is None:
            continue

        xmin = int(float(box.findtext('xmin', default='0')))
        ymin = int(float(box.findtext('ymin', default='0')))
        xmax = int(float(box.findtext('xmax', default='0')))
        ymax = int(float(box.findtext('ymax', default='0')))

        box_w = max(1, xmax - xmin + 1)
        box_h = max(1, ymax - ymin + 1)

        records.append({
            'image_id': image_id,
            'split': split_name,
            'cls': cls,
            'class_id': CLASS_TO_ID[cls],
            'xmin': xmin,
            'ymin': ymin,
            'xmax': xmax,
            'ymax': ymax,
            'box_w': box_w,
            'box_h': box_h,
            'box_area': box_w * box_h,
            'difficult': int(obj.findtext('difficult', default='0')),
            'truncated': int(obj.findtext('truncated', default='0')),
            'occluded': int(obj.findtext('occluded', default='0')),
            'width': width,
            'height': height,
        })

    image_meta = {
        'image_id': image_id,
        'split': split_name,
        'width': width,
        'height': height,
    }
    return image_meta, records



def build_split_tables(split_name: str, image_ids: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    image_rows, object_rows = [], []
    ann_dir = ANNOTATION_ROOT / split_name

    for image_id in image_ids:
        xml_path = ann_dir / f'{image_id}.xml'
        if not xml_path.exists():
            image_rows.append({'image_id': image_id, 'split': split_name, 'width': np.nan, 'height': np.nan})
            continue

        image_meta, rows = parse_annotation(xml_path, image_id, split_name)
        image_rows.append(image_meta)
        object_rows.extend(rows)

    images_df = pd.DataFrame(image_rows).drop_duplicates(subset=['image_id', 'split'])
    objects_df = pd.DataFrame(object_rows)

    if not objects_df.empty:
        objects_df['box_w'] = (objects_df['xmax'] - objects_df['xmin'] + 1).clip(lower=1)
        objects_df['box_h'] = (objects_df['ymax'] - objects_df['ymin'] + 1).clip(lower=1)
        objects_df['box_area'] = objects_df['box_w'] * objects_df['box_h']
        objects_df['aspect_ratio'] = objects_df['box_w'] / objects_df['box_h']
        image_area = (objects_df['width'] * objects_df['height']).replace(0, np.nan)
        objects_df['norm_area'] = objects_df['box_area'] / image_area

    return images_df, objects_df


train_ids = load_split_ids('train')
val_ids = load_split_ids('val')
dev_test_ids = load_split_ids('dev_test')
test_ids = load_split_ids('test')

train_images, train_objects = build_split_tables('train', train_ids)
val_images, val_objects = build_split_tables('val', val_ids)
dev_test_images, dev_test_objects = build_split_tables('dev_test', dev_test_ids)
test_images, test_objects = build_split_tables('test', test_ids)

train_model_df = train_objects[(train_objects['xmax'] >= train_objects['xmin']) & (train_objects['ymax'] >= train_objects['ymin'])].copy()
val_model_df = val_objects[(val_objects['xmax'] >= val_objects['xmin']) & (val_objects['ymax'] >= val_objects['ymin'])].copy()
dev_test_model_df = dev_test_objects[(dev_test_objects['xmax'] >= dev_test_objects['xmin']) & (dev_test_objects['ymax'] >= dev_test_objects['ymin'])].copy()

train_model_df_ignoring_difficult = train_model_df[train_model_df['difficult'] == 0].reset_index(drop=True)
val_model_df_ignoring_difficult = val_model_df[val_model_df['difficult'] == 0].reset_index(drop=True)
dev_test_model_df_ignoring_difficult = dev_test_model_df[dev_test_model_df['difficult'] == 0].reset_index(drop=True)

print('Split counts:', len(train_ids), len(val_ids), len(dev_test_ids), len(test_ids))
print('train-val overlap:', len(set(train_ids) & set(val_ids)))
print('train-dev_test overlap:', len(set(train_ids) & set(dev_test_ids)))
print('val-dev_test overlap:', len(set(val_ids) & set(dev_test_ids)))
print('train-test overlap:', len(set(train_ids) & set(test_ids)))
print('val-test overlap:', len(set(val_ids) & set(test_ids)))
print('dev_test-test overlap:', len(set(dev_test_ids) & set(test_ids)))

display(train_model_df_ignoring_difficult.head())

## 4. Exploratory Data Analysis

This section validates dataset quality before training: class balance, annotation flags, and bounding-box geometry.

These checks justify design choices for training mode, augmentation, and evaluation focus.

In [ ]:
trainval_images = pd.concat([train_images, val_images], ignore_index=True).drop_duplicates(subset=['image_id'])
trainval_objects = pd.concat([train_objects, val_objects], ignore_index=True)

if trainval_objects.empty:
    raise RuntimeError('No train/val annotations found. Check annotation paths and split files.')

eda_df = trainval_objects.copy()
obj_counts = eda_df.groupby('cls').size().reindex(VOC_CLASSES, fill_value=0).rename('object_count')
img_counts = eda_df.groupby('cls')['image_id'].nunique().reindex(VOC_CLASSES, fill_value=0).rename('image_count')
class_stats = pd.concat([img_counts, obj_counts], axis=1).sort_values('object_count', ascending=False)

class_stats

In [ ]:
plt.figure(figsize=(12, 5))
order = class_stats.index.tolist()
sns.barplot(data=class_stats.reset_index(), x='cls', y='object_count', order=order, color='#2a9d8f')
plt.xticks(rotation=45, ha='right')
plt.title('Object Count per Class (train + val)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
sns.barplot(data=class_stats.reset_index(), x='cls', y='image_count', order=order, color='#e76f51')
plt.xticks(rotation=45, ha='right')
plt.title('Image Count per Class (train + val)')
plt.tight_layout()
plt.show()

### EDA: Augmentation Preview

This quick diagnostic visualizes random training images before and after augmentation, with bounding boxes overlaid in both views.

Use it to sanity-check whether spatial transforms (for example horizontal flip) keep boxes aligned with the objects.

In [ ]:
def get_transforms(train: bool):
    transforms = []
    transforms.append(T.ToImage())
    transforms.append(T.ToDtype(torch.float32, scale=True))
    
    if train:
        transforms.append(T.RandomHorizontalFlip(p=0.5))
        transforms.append(T.ColorJitter(brightness=0.25, contrast=0.25))
    
    transforms.append(T.SanitizeBoundingBoxes())
    return T.Compose(transforms)

In [ ]:
class VOCDetectionDataset(Dataset):
    def __init__(self, image_dir: Path, image_ids: list[str], objects_df: pd.DataFrame, transforms=None):
        self.image_dir = image_dir
        self.image_ids = image_ids
        self.objects_df = objects_df
        self.transforms = transforms

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        image_path = self.image_dir / f'{img_id}.jpg'
        image = Image.open(image_path).convert('RGB')
        
        rows = self.objects_df[self.objects_df['image_id'] == img_id]
        if rows.empty:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(rows[['xmin', 'ymin', 'xmax', 'ymax']].to_numpy(), dtype=torch.float32)
            # IMPORTANT: Faster R-CNN labels must be 1-indexed (0=background)
            labels = torch.tensor(rows['class_id'].to_numpy(), dtype=torch.int64) + 1
        
        # Wrap in tv_tensors
        image = tv_tensors.Image(image)
        target = {
            'boxes': tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=image.shape[-2:]),
            'labels': labels,
            'image_id': img_id
        }

        if self.transforms:
            image, target = self.transforms(image, target)

        return image, target

## 5. Faster R-CNN Setup and Training Utilities

This section defines the detector architecture, preprocessing presets, training modes, checkpointing, and mAP@0.5 evaluation utilities.

Training modes implemented:
- Head-only fine-tuning
- Full fine-tuning
- Incremental unfreezing

In [ ]:
# --- Missing Training & Dataset Utilities ---
import time
import gc
import numpy as np
from tqdm.auto import tqdm
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

DEFAULT_BASE_RCNN_MODEL = 'resnet50_v2'
BASE_RCNN_MODELS = {
    'resnet50_v2': {
        'factory': fasterrcnn_resnet50_fpn_v2,
        'weights': FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    }
}

def collate_fn(batch):
    return tuple(zip(*batch))

def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def ensure_prediction_schema(df):
    required = ['image_id', 'xmin', 'ymin', 'xmax', 'ymax', 'score', 'class_id']
    for col in required:
        if col not in df.columns:
            df[col] = None
    return df

def build_faster_rcnn_model(num_classes: int, base_model_name: str = DEFAULT_BASE_RCNN_MODEL) -> torch.nn.Module:
    model_spec = BASE_RCNN_MODELS[base_model_name]
    model = model_spec['factory'](weights=model_spec['weights'])
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

def freeze_all_parameters(model: torch.nn.Module) -> None:
    for parameter in model.parameters():
        parameter.requires_grad = False

def set_head_only_trainable(model: torch.nn.Module) -> None:
    freeze_all_parameters(model)
    for parameter in model.roi_heads.box_predictor.parameters():
        parameter.requires_grad = True

def set_full_finetune_trainable(model: torch.nn.Module) -> None:
    for parameter in model.parameters():
        parameter.requires_grad = True

def set_incremental_trainable(model, epoch_index, total_layers=50, layers_per_epoch=10):
    """
    Gradually unfreezes the ResNet backbone stages.
    Stages: layer4 (deepest), layer3, layer2, layer1.
    """
    # Freeze everything first
    for param in model.parameters():
        param.requires_grad = False
    
    # Always train the detection head
    for param in model.roi_heads.box_predictor.parameters():
        param.requires_grad = True
    
    # Define backbone stages
    try:
        body = model.backbone.body
        stages = [body.layer4, body.layer3, body.layer2, body.layer1]
        
        # Unfreeze stages from top to bottom
        unfreeze_count = min(epoch_index + 1, len(stages))
        active_stages = []
        for i in range(unfreeze_count):
            for param in stages[i].parameters():
                param.requires_grad = True
            active_stages.append(f"layer{4-i}")
            
        # Also unfreeze FPN if we started unfreezing backbone
        if unfreeze_count > 0:
            for param in model.backbone.fpn.parameters():
                param.requires_grad = True
            active_stages.append("fpn")
            
        return active_stages
    except AttributeError:
        # Fallback
        for param in model.parameters():
            param.requires_grad = True
        return ["all"]

def count_trainable_parameters(model: torch.nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

DATASET_CACHE = {}

def build_datasets_and_loaders(preprocessing_mode: str, batch_size: int):
    cache_key = (preprocessing_mode, batch_size)
    if cache_key in DATASET_CACHE:
        return DATASET_CACHE[cache_key]

    train_transform = get_transforms(train=True)
    eval_transform = get_transforms(train=False)

    train_ds = VOCDetectionDataset(TRAIN_IMAGE_DIR, train_ids, train_model_df_ignoring_difficult, transforms=train_transform)
    val_ds = VOCDetectionDataset(VAL_IMAGE_DIR, val_ids, val_model_df_ignoring_difficult, transforms=eval_transform)
    dev_test_ds = VOCDetectionDataset(DEV_TEST_IMAGE_DIR, dev_test_ids, dev_test_model_df_ignoring_difficult, transforms=eval_transform)
    test_ds = VOCDetectionDataset(TEST_IMAGE_DIR, test_ids, test_objects, transforms=eval_transform)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    dev_test_loader = DataLoader(dev_test_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    DATASET_CACHE[cache_key] = (train_ds, val_ds, dev_test_ds, test_ds, train_loader, val_loader, dev_test_loader, test_loader)
    return DATASET_CACHE[cache_key]

def train_one_epoch(model, loader, optimizer, device, epoch_idx, total_epochs):
    model.train()
    total_loss = 0
    start_time = time.perf_counter()
    batch_times = []
    
    for images, targets in tqdm(loader, desc=f"Epoch {epoch_idx+1}/{total_epochs}", leave=False, dynamic_ncols=True):
        batch_start = time.perf_counter()
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in targets]
        
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        total_loss += losses.item()
        batch_times.append(time.perf_counter() - batch_start)
        
    return total_loss / len(loader), time.perf_counter() - start_time, np.mean(batch_times)

def predict_to_df(model, loader, split_name, score_threshold=0.05):
    model.eval()
    all_preds = []
    start_time = time.perf_counter()
    
    with torch.no_grad():
        for images, targets in tqdm(loader, desc=f"Predicting {split_name}", leave=False, dynamic_ncols=True):
            img_ids = [t['image_id'] for t in targets]
            images = list(image.to(DEVICE) for image in images)
            outputs = model(images)
            
            for img_id, output in zip(img_ids, outputs):
                boxes = output['boxes'].cpu().numpy()
                scores = output['scores'].cpu().numpy()
                labels = output['labels'].cpu().numpy()
                
                keep = scores >= score_threshold
                boxes, scores, labels = boxes[keep], scores[keep], labels[keep]
                
                for box, score, label in zip(boxes, scores, labels):
                    all_preds.append({
                        'image_id': img_id,
                        'split': split_name,
                        'class_id': int(label) - 1,
                        'cls': VOC_CLASSES[int(label) - 1],
                        'xmin': box[0], 'ymin': box[1], 'xmax': box[2], 'ymax': box[3],
                        'score': score
                    })
                    
    return pd.DataFrame(all_preds), time.perf_counter() - start_time

def save_checkpoint(model, optimizer, epoch, path, extra_state=None):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'seed': SEED,
    }
    if extra_state: checkpoint.update(extra_state)
    torch.save(checkpoint, path)

def load_checkpoint(path, map_location=DEVICE):
    if not path or not Path(path).exists(): return None
    return torch.load(path, map_location=map_location)


In [ ]:
def evaluate_map50(pred_df, gt_df, class_names=None):
    from torchmetrics.detection import MeanAveragePrecision
    metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox', class_metrics=False)
    
    # If evaluating a single class (e.g. 'person'), filter the dataframes first
    # to ensure we compute AP for that class correctly.
    if class_names and len(class_names) == 1:
        target_name = class_names[0]
        target_id = CLASS_TO_ID.get(target_name)
        if target_id is not None:
            pred_df = pred_df[pred_df['class_id'] == target_id]
            gt_df = gt_df[gt_df['class_id'] == target_id]

    all_img_ids = sorted(list(set(pred_df['image_id']) | set(gt_df['image_id'])))
    for img_id in all_img_ids:
        p = pred_df[pred_df['image_id'] == img_id]
        g = gt_df[gt_df['image_id'] == img_id]
        
        if p.empty:
            preds = [{'boxes': torch.zeros((0, 4), dtype=torch.float32), 'scores': torch.zeros((0,), dtype=torch.float32), 'labels': torch.zeros((0,), dtype=torch.int64)}]
        else:
            preds = [{
                'boxes': torch.tensor(p[['xmin', 'ymin', 'xmax', 'ymax']].values.astype(float), dtype=torch.float32),
                'scores': torch.tensor(p['score'].values.astype(float), dtype=torch.float32),
                'labels': torch.tensor(p['class_id'].values.astype(int), dtype=torch.int64) + 1
            }]
            
        if g.empty:
            gts = [{'boxes': torch.zeros((0, 4), dtype=torch.float32), 'labels': torch.zeros((0,), dtype=torch.int64)}]
        else:
            gts = [{
                'boxes': torch.tensor(g[['xmin', 'ymin', 'xmax', 'ymax']].values.astype(float), dtype=torch.float32),
                'labels': torch.tensor(g['class_id'].values.astype(int), dtype=torch.int64) + 1
            }]
        metric.update(preds, gts)
        
    results = metric.compute()
    return {'map50': float(results['map_50'].item())}

In [ ]:
def print_per_class_map(results, class_names, title="Per-Class mAP@[.5:.95]"):
    """
    Prints a formatted table of mAP scores for each class.
    """
    if 'map_per_class' not in results:
        return
    
    scores = results['map_per_class']
    if isinstance(scores, torch.Tensor):
        scores = scores.tolist()
        
    print(f"\n=== {title} ===")
    print(f"{'ID':<3} | {'Class Name':<15} | {'mAP':<6}")
    print("-" * 30)
    for i, (name, score) in enumerate(zip(class_names, scores)):
        print(f"{i+1:<3} | {name:<15} | {score:.4f}")
    print("-" * 30)


def evaluate_map_torchmetrics(model, loader, device=DEVICE, class_metrics=False):
    """
    Calculates mAP@0.5 using torchmetrics.detection.MeanAveragePrecision.
    Faster R-CNN model outputs labels 1-indexed (0=background).
    Ground truth from VOCDetectionDataset is 0-indexed, so we add 1 for alignment.
    """
    model.eval()
    metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox', class_metrics=class_metrics)
    metric.to(device)
    
    with torch.no_grad():
        for images, targets in tqdm(loader, desc="Evaluating", leave=False, dynamic_ncols=True):
            images = [img.to(device) for img in images]
            outputs = model(images)
            
            preds = []
            for out in outputs:
                preds.append({
                    'boxes': out['boxes'].detach().cpu(),
                    'scores': out['scores'].detach().cpu(),
                    'labels': out['labels'].detach().cpu()
                })
            
            gts = []
            for t in targets:
                gts.append({
                    'boxes': t['boxes'].detach().cpu(),
                    'labels': t['labels'].detach().cpu()
                })
            
            metric.update(preds, gts)
            
    results = metric.compute()
    # Clean up tensor types for easier dict access
    return {k: v.item() if v.numel() == 1 else v.tolist() for k, v in results.items()}


def train_and_evaluate_experiment(experiment_name: str, config: dict) -> tuple[dict, pd.DataFrame]:
    train_ds, val_ds, dev_test_ds, test_ds, train_loader, val_loader, dev_test_loader, test_loader = build_datasets_and_loaders(
        config['preprocessing_mode'],
        config['batch_size'],
    )

    model = build_faster_rcnn_model(len(VOC_CLASSES) + 1, base_model_name=config.get('base_model_name', DEFAULT_BASE_RCNN_MODEL))
    model.to(DEVICE)
    
    # Standardizing to SGD as per torchvision tutorial recommendations for Faster R-CNN
    optimizer = torch.optim.SGD(
        model.parameters(), 
        lr=config['lr'], 
        momentum=0.9, 
        weight_decay=config['weight_decay']
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['epochs'])

    if config['training_mode'] == 'head_only':
        set_head_only_trainable(model)
    elif config['training_mode'] == 'full_finetune':
        set_full_finetune_trainable(model)
    elif config['training_mode'] == 'incremental':
        set_head_only_trainable(model)
    else:
        raise ValueError(f"Unknown training_mode: {config['training_mode']}")

    trainable_params = count_trainable_parameters(model)
    total_params = sum(parameter.numel() for parameter in model.parameters())
    print(f"\n=== Experiment: {experiment_name} ===")
    print(f"Base model: {config.get('base_model_name', DEFAULT_BASE_RCNN_MODEL)}")
    print(f"Preprocessing: {config['preprocessing_mode']} | Training: {config['training_mode']}")
    print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")

    checkpoint_dir = EXPERIMENT_RUNS_DIR / experiment_name
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = checkpoint_dir / 'best_checkpoint.pt'

    epoch_rows = []
    best_map50 = -1.0
    patience = 3
    epochs_without_improvement = 0
    total_train_start = time.perf_counter()

    for epoch in range(config['epochs']):
        if config['training_mode'] == 'incremental':
            active_layers = set_incremental_trainable(
                model,
                epoch_index=epoch,
                total_layers=config.get('incremental_layers_total', 50),
                layers_per_epoch=config.get('incremental_layers_per_epoch', 10),
            )
            print(f"Epoch {epoch + 1}: incremental backbone layers active = {len(active_layers)}")

        epoch_loss, epoch_seconds, batch_seconds = train_one_epoch(model, train_loader, optimizer, DEVICE, epoch_idx=epoch, total_epochs=config['epochs'])
        scheduler.step()
        
        # Using efficient torchmetrics evaluation instead of writing DataFrames every epoch
        val_results = evaluate_map_torchmetrics(model, val_loader, device=DEVICE)
        val_map50 = val_results['map_50']
        
        dev_test_results = evaluate_map_torchmetrics(model, dev_test_loader, device=DEVICE)
        dev_test_map50 = dev_test_results['map_50']

        epoch_rows.append({
            'experiment': experiment_name,
            'epoch': epoch + 1,
            'loss': epoch_loss,
            'epoch_seconds': epoch_seconds,
            'batch_seconds': batch_seconds,
            'val_map50': val_map50,
            'dev_test_map50': dev_test_map50,
        })

        print(
            f"Epoch {epoch + 1}/{config['epochs']} | loss={epoch_loss:.4f} | "
            f"train_time={epoch_seconds:.1f}s | val_mAP@0.5={val_map50:.4f} | dev_test_mAP@0.5={dev_test_map50:.4f}"
        )

        if val_map50 >= best_map50:
            best_map50 = val_map50
            epochs_without_improvement = 0
            save_checkpoint(model, optimizer, epoch, checkpoint_path, extra_state={'experiment': experiment_name})
            print(f"Saved best checkpoint to {checkpoint_path}")
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print(f"Early stopping triggered after {epoch + 1} epochs")
            break

    total_train_seconds = time.perf_counter() - total_train_start

    # Load best model for final evaluation and prediction export
    checkpoint = load_checkpoint(checkpoint_path)
    if checkpoint is not None:
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        del checkpoint

    # Final predictions with best model (only once at the end)
    best_threshold = 0.05 
    
    val_pred_df, val_inference_seconds = predict_to_df(model, val_loader, split_name='val', score_threshold=best_threshold)
    dev_test_pred_df, dev_test_inference_seconds = predict_to_df(model, dev_test_loader, split_name='dev_test', score_threshold=best_threshold)
    test_pred_df, test_inference_seconds = predict_to_df(model, test_loader, split_name='test', score_threshold=best_threshold)

    # Final metrics using best model (with class_metrics=True)
    val_results = evaluate_map_torchmetrics(model, val_loader, device=DEVICE, class_metrics=True)
    dev_test_results = evaluate_map_torchmetrics(model, dev_test_loader, device=DEVICE, class_metrics=True)
    test_results = evaluate_map_torchmetrics(model, test_loader, device=DEVICE, class_metrics=True)

    val_map50 = float(val_results['map_50'])
    dev_test_map50 = float(dev_test_results['map_50'])
    test_map50 = float(test_results['map_50'])

    val_path = checkpoint_dir / f'{experiment_name}_val_predictions.csv'
    dev_test_path = checkpoint_dir / f'{experiment_name}_dev_test_predictions.csv'
    test_path = checkpoint_dir / f'{experiment_name}_test_predictions.csv'
    history_path = checkpoint_dir / f'{experiment_name}_history.csv'

    val_pred_df.to_csv(val_path, index=False)
    dev_test_pred_df.to_csv(dev_test_path, index=False)
    test_pred_df.to_csv(test_path, index=False)
    history_df = pd.DataFrame(epoch_rows)
    history_df.to_csv(history_path, index=False)

    summary = {
        'experiment': experiment_name,
        'base_model_name': config.get('base_model_name', DEFAULT_BASE_RCNN_MODEL),
        'preprocessing_mode': config['preprocessing_mode'],
        'training_mode': config['training_mode'],
        'epochs': config['epochs'],
        'batch_size': config['batch_size'],
        'trainable_params': trainable_params,
        'total_params': total_params,
        'best_epoch_val_map50': best_map50,
        'tuned_val_map50': val_map50,
        'tuned_dev_test_map50': dev_test_map50,
        'tuned_test_map50': test_map50,
        'best_threshold': best_threshold,
        'train_seconds': total_train_seconds,
        'val_inference_seconds': val_inference_seconds,
        'dev_test_inference_seconds': dev_test_inference_seconds,
        'test_inference_seconds': test_inference_seconds,
        'val_seconds_per_image': val_inference_seconds / max(1, len(val_ds)),
        'dev_test_seconds_per_image': dev_test_inference_seconds / max(1, len(dev_test_ds)),
        'test_seconds_per_image': test_inference_seconds / max(1, len(test_ds)),
        'history_path': str(history_path),
        'val_predictions_path': str(val_path),
        'dev_test_predictions_path': str(dev_test_path),
        'test_predictions_path': str(test_path),
        'checkpoint_path': str(checkpoint_path),
    }

    print(
        f"Completed {experiment_name} | val_mAP@0.5={val_map50:.4f} | dev_test_mAP@0.5={dev_test_map50:.4f} | "
        f"test_mAP@0.5={test_map50:.4f} | train_time={total_train_seconds:.1f}s"
    )

    # Detailed per-class report for test set
    print_per_class_map(test_results, VOC_CLASSES, title=f"Test Set Per-Class mAP for {experiment_name}")

    del val_pred_df, dev_test_pred_df, test_pred_df, epoch_rows
    del model, optimizer, train_ds, val_ds, dev_test_ds, test_ds, train_loader, val_loader, dev_test_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary, history_df


### 5.1 Note on Class-Imbalance

Faster R-CNN handles class imbalance natively through ROI sampling. Custom classification loss weighting is not required for standard VOC fine-tuning.

## 6. Experiment Suite (3 Explicit Runs)

Each experiment is now executed in its own code cell (no for-loop), which improves reproducibility, debugging, and report presentation.

Run the setup cell first, then execute the five experiments below.

In [ ]:
EXPERIMENT_LIBRARY = {
    'head_only_baseline': {
        'base_model_name': DEFAULT_BASE_RCNN_MODEL,
        'preprocessing_mode': 'baseline',
        'training_mode': 'head_only',
        'epochs': 10,
        'batch_size': 2,
        'lr': 1e-4,
        'weight_decay': 1e-4,
    },
    'augmentation_head_only': {
        'base_model_name': DEFAULT_BASE_RCNN_MODEL,
        'preprocessing_mode': 'augmentation',
        'training_mode': 'head_only',
        'epochs': 10,
        'batch_size': 2,
        'lr': 1e-4,
        'weight_decay': 1e-4,
    },
    'preprocessing_noise_head_only': {
        'base_model_name': DEFAULT_BASE_RCNN_MODEL,
        'preprocessing_mode': 'preprocessing_noise',
        'training_mode': 'head_only',
        'epochs': 10,
        'batch_size': 2,
        'lr': 1e-4,
        'weight_decay': 1e-4,
    },
    'incremental_unfreezing': {
        'base_model_name': DEFAULT_BASE_RCNN_MODEL,
        'preprocessing_mode': 'baseline',
        'training_mode': 'incremental',
        'epochs': 15,
        'batch_size': 2,
        'lr': 1e-4,
        'weight_decay': 1e-4,
        'incremental_layers_total': 4,
        'incremental_layers_per_epoch': 1,
    },
}

In [ ]:
EXPERIMENT_RESULTS = []
EXPERIMENT_HISTORY = {}

def run_single_experiment(experiment_name: str):
    if experiment_name not in EXPERIMENT_LIBRARY:
        raise KeyError(f'Unknown experiment: {experiment_name}')

    summary, history_df = train_and_evaluate_experiment(
        experiment_name,
        EXPERIMENT_LIBRARY[experiment_name],
    )
    EXPERIMENT_RESULTS.append(summary)
    EXPERIMENT_HISTORY[experiment_name] = history_df
    return summary, history_df

print('Experiment containers are ready. Run the five experiment cells below in order.')

### 6.1 Experiment 1 - Head-Only Baseline

Objective: establish a lightweight baseline by training only the detection head while keeping the backbone frozen.

In [ ]:
summary_head_only, history_head_only = run_single_experiment('head_only_baseline')
display(pd.DataFrame([summary_head_only]))
display(history_head_only.tail())

### 6.2 Experiment 2 - Augmentation Head-Only


In [ ]:
summary_preproc_ft, history_preproc_ft = run_single_experiment('augmentation_head_only')
display(pd.DataFrame([summary_preproc_ft]))
display(history_preproc_ft.tail())

### 6.3 Experiment 3 - Preprocessing + Noise Head-Only


In [ ]:
summary_preproc_noise_ft, history_preproc_noise_ft = run_single_experiment('preprocessing_noise_head_only')
display(pd.DataFrame([summary_preproc_noise_ft]))
display(history_preproc_noise_ft.tail())

### 6.3.5 Experiment 4 - Incremental Backbone Unfreezing

Objective: evaluate whether unfreezing the backbone stages gradually improves the mAP beyond the head-only baseline.

In [ ]:
summary_incremental, history_incremental = run_single_experiment('incremental_unfreezing')
display(pd.DataFrame([summary_incremental]))
display(history_incremental.tail())

### 6.4 Aggregate Experiment Results


In [ ]:
results_df = pd.DataFrame(EXPERIMENT_RESULTS).sort_values('tuned_val_map50', ascending=False).reset_index(drop=True)
results_path = EXPERIMENT_RUNS_DIR / 'experiment_summary.csv'
results_df.to_csv(results_path, index=False)

display(
    results_df[
        [
            'experiment',
            'base_model_name',
            'preprocessing_mode',
            'training_mode',
            'epochs',
            'trainable_params',
            'best_epoch_val_map50',
            'tuned_val_map50',
            'tuned_dev_test_map50',
            'tuned_test_map50',
            'best_threshold',
            'train_seconds',
            'val_inference_seconds',
            'dev_test_inference_seconds',
            'test_inference_seconds',
        ]
    ]
)
print('Saved experiment summary to:', results_path)

### 6.5 Per-Experiment Visualisation


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Per-class AP@0.5 comparison
per_class_df[['val_ap50', 'test_ap50']].plot(kind='bar', ax=axes[0], width=0.8)
axes[0].set_title(f'Per-class AP@0.5 ({best_experiment})')
axes[0].set_ylabel('AP@0.5')
axes[0].set_ylim(0, 1.05)
axes[0].grid(axis='y', linestyle='--', alpha=0.7)
axes[0].legend(['Validation', 'Test'])

# Plot 2: Overall mAP comparison
splits = ['val', 'dev_test', 'test']
maps = [overall_val_map, overall_dev_test_map, overall_test_map]
bars = axes[1].bar(splits, maps, color=['blue', 'orange', 'green'], alpha=0.7)
axes[1].set_title('Overall mAP@0.5 by Split')
axes[1].set_ylabel('mAP@0.5')
axes[1].set_ylim(0, 1.05)
axes[1].grid(axis='y', linestyle='--', alpha=0.7)

for bar in bars:
    yval = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2, yval + 0.02, f'{yval:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()


## 7. Evaluation and Visualisation (All Classes)

This section evaluates the best performing model across all 20 PASCAL VOC classes. We calculate AP@0.5 for each class individually and report the overall mean AP (mAP@0.5) across the `val`, `dev_test`, and `test` splits.

In [ ]:
if 'EXPERIMENT_RESULTS' not in globals() or not EXPERIMENT_RESULTS:
raise RuntimeError('Run Experiment Suite first.')

# Load best experiment model
best_summary = sorted(EXPERIMENT_RESULTS, key=lambda x: x['tuned_val_map50'], reverse=True)[0]
experiment_name = best_summary['experiment']
checkpoint_path = Path(best_summary['checkpoint_path'])

print(f"Evaluating best model from: {experiment_name}")
model = build_faster_rcnn_model(len(VOC_CLASSES) + 1)
model.to(DEVICE)
checkpoint = load_checkpoint(checkpoint_path)
model.load_state_dict(checkpoint['model_state_dict'])

_, val_ds, dev_test_ds, test_ds, _, val_loader, dev_test_loader, test_loader = build_datasets_and_loaders(
best_summary['preprocessing_mode'],
best_summary['batch_size']
)

# Multi-class evaluation using torchmetrics
val_results = evaluate_map_torchmetrics(model, val_loader, class_metrics=True)
dev_test_results = evaluate_map_torchmetrics(model, dev_test_loader, class_metrics=True)
test_results = evaluate_map_torchmetrics(model, test_loader, class_metrics=True)

# Format into a clean DataFrame
per_class_data = []
for i, cls_name in enumerate(VOC_CLASSES):
per_class_data.append({
'class': cls_name,
'val_ap50': val_results['map_per_class'][i],
'dev_test_ap50': dev_test_results['map_per_class'][i],
'test_ap50': test_results['map_per_class'][i]
})

per_class_df = pd.DataFrame(per_class_data).set_index('class')
display(per_class_df.round(4))

overall_val_map = val_results['map_50']
overall_dev_test_map = dev_test_results['map_50']
overall_test_map = test_results['map_50']

print(f"Overall mAP@0.5 (val):      {overall_val_map:.4f}")
print(f"Overall mAP@0.5 (dev_test): {overall_dev_test_map:.4f}")
print(f"Overall mAP@0.5 (test):     {overall_test_map:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Per-class AP@0.5 comparison
per_class_df[['val_ap50', 'test_ap50']].plot(kind='bar', ax=axes[0], width=0.8)
axes[0].set_title(f'Per-class AP@0.5 ({best_experiment})')
axes[0].set_ylabel('AP@0.5')
axes[0].set_ylim(0, 1.05)
axes[0].grid(axis='y', linestyle='--', alpha=0.7)
axes[0].legend(['Validation', 'Test'])

# Plot 2: Overall mAP comparison
splits = ['val', 'dev_test', 'test']
maps = [overall_val_map, overall_dev_test_map, overall_test_map]
bars = axes[1].bar(splits, maps, color=['blue', 'orange', 'green'], alpha=0.7)
axes[1].set_title('Overall mAP@0.5 by Split')
axes[1].set_ylabel('mAP@0.5')
axes[1].set_ylim(0, 1.05)
axes[1].grid(axis='y', linestyle='--', alpha=0.7)

for bar in bars:
    yval = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2, yval + 0.02, f'{yval:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()


## 8. Predict and Export

Once training completes, this section reports the best experiment and exposes the prediction CSV paths for downstream ensemble/server evaluation workflows.

In [ ]:
if 'EXPERIMENT_RESULTS' not in globals() or not EXPERIMENT_RESULTS:
    raise RuntimeError('Run the experiment suite first so the notebook has results to export.')

results_df = pd.DataFrame(EXPERIMENT_RESULTS).sort_values('tuned_val_map50', ascending=False).reset_index(drop=True)
best_row = results_df.iloc[0]
best_experiment = best_row['experiment']

print('Best experiment:', best_experiment)
print('Preprocessing mode:', best_row['preprocessing_mode'])
print('Training mode:', best_row['training_mode'])
print('Validation mAP@0.5:', round(float(best_row['tuned_val_map50']), 4))
print('Best threshold:', f"{float(best_row['best_threshold']):.2f}")
print('Train time (seconds):', round(float(best_row['train_seconds']), 1))
print('Validation inference time (seconds):', round(float(best_row['val_inference_seconds']), 1))
print('Test inference time (seconds):', round(float(best_row['test_inference_seconds']), 1))
print('Validation predictions:', best_row['val_predictions_path'])
print('Test predictions:', best_row['test_predictions_path'])
print('History CSV:', best_row['history_path'])
print('Checkpoint:', best_row['checkpoint_path'])

display(results_df)
display(EXPERIMENT_HISTORY[best_experiment].tail())

## 9. Base Model Comparison (No Fine-Tuning)

This section evaluates a few pretrained TorchVision Faster R-CNN variants on the same VOC validation/test splits.

Use the validation split to choose the best architecture or fine-tuning recipe. That is not data leakage. It becomes leakage only if the test split influences model selection or hyperparameter tuning.

A safe workflow is:
- compare candidate RCNN variants on train/val only
- pick the best candidate using validation metrics
- fine-tune that candidate
- evaluate once on the untouched test split, or submit to the official VOC server

This section also compares the best fine-tuned experiment against a zero-shot COCO baseline so you can see how much gain comes from VOC adaptation versus the pretrained detector alone.

In [ ]:
from pathlib import Path
import time

import torchvision
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights

if 'results_df' not in globals() or results_df.empty:
    raise RuntimeError('Run Section 6 and Section 8 first so the best fine-tuned experiment is available.')

best_row = results_df.iloc[0]
best_experiment = str(best_row['experiment'])

best_val_path = Path(best_row['val_predictions_path'])
best_dev_test_path = Path(best_row['dev_test_predictions_path']) if 'dev_test_predictions_path' in best_row and pd.notna(best_row['dev_test_predictions_path']) else None
best_test_path = Path(best_row['test_predictions_path'])
if not best_val_path.exists() or not best_test_path.exists():
    raise FileNotFoundError('Best experiment prediction files were not found. Re-run Section 8 to regenerate them.')

best_val_pred = pd.read_csv(best_val_path)
best_dev_test_pred = pd.read_csv(best_dev_test_path) if best_dev_test_path and best_dev_test_path.exists() else pd.DataFrame(columns=PREDICTION_COLUMNS)
best_test_pred = pd.read_csv(best_test_path)

# Build evaluation loaders with baseline preprocessing for a fair no-finetuning comparison.
_, val_ds_eval, dev_test_ds_eval, test_ds_eval, _, val_loader_eval, dev_test_loader_eval, test_loader_eval = build_datasets_and_loaders(
    preprocessing_mode='baseline',
    batch_size=2,
)

weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
base_model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=weights)
base_model.to(DEVICE)
base_model.eval()

coco_categories = list(weights.meta['categories'])
coco_name_to_voc = {
    'airplane': 'aeroplane',
    'bicycle': 'bicycle',
    'bird': 'bird',
    'boat': 'boat',
    'bottle': 'bottle',
    'bus': 'bus',
    'car': 'car',
    'cat': 'cat',
    'chair': 'chair',
    'cow': 'cow',
    'dining table': 'diningtable',
    'dog': 'dog',
    'horse': 'horse',
    'motorcycle': 'motorbike',
    'person': 'person',
    'potted plant': 'pottedplant',
    'sheep': 'sheep',
    'couch': 'sofa',
    'train': 'train',
    'tv': 'tvmonitor',
}

coco_label_to_voc = {}
for i, coco_name in enumerate(coco_categories):
    voc_name = coco_name_to_voc.get(coco_name)
    if voc_name is not None:
        coco_label_to_voc[i + 1] = voc_name


def predict_base_model_to_voc_df(model, loader, split_name: str, score_threshold: float = 0.05, max_detections: int = 50):
    rows = []
    sync_cuda()
    start_time = time.perf_counter()

    with torch.no_grad():
        for images, targets in loader:
            image_ids = [target['image_id'] for target in targets]
            images_gpu = [image.to(DEVICE) for image in images]
            outputs = model(images_gpu)
            sync_cuda()

            for image_id, output in zip(image_ids, outputs):
                scores = output['scores'].detach().cpu().numpy()
                boxes = output['boxes'].detach().cpu().numpy()
                labels = output['labels'].detach().cpu().numpy()

                keep = scores >= score_threshold
                boxes = boxes[keep][:max_detections]
                scores_kept = scores[keep][:max_detections]
                labels_kept = labels[keep][:max_detections]

                for box, score, coco_label in zip(boxes, scores_kept, labels_kept):
                    voc_name = coco_label_to_voc.get(int(coco_label))
                    if voc_name is None:
                        continue
                    rows.append({
                        'image_id': image_id,
                        'class_id': int(CLASS_TO_ID[voc_name]),
                        'cls': voc_name,
                        'xmin': float(box[0]),
                        'ymin': float(box[1]),
                        'xmax': float(box[2]),
                        'ymax': float(box[3]),
                        'score': float(score),
                    })

    sync_cuda()
    elapsed = time.perf_counter() - start_time
    pred_df = ensure_prediction_schema(pd.DataFrame(rows), source_model='fasterrcnn_coco_base', split_name=split_name)
    return pred_df, elapsed


candidate_thresholds = [round(x, 2) for x in np.arange(0.05, 0.55, 0.05)]
threshold_rows = []
best_base_threshold = candidate_thresholds[0]
best_base_map50 = -1.0

for threshold in candidate_thresholds:
    base_val_pred_tmp, val_seconds_tmp = predict_base_model_to_voc_df(
        base_model,
        val_loader_eval,
        split_name='val',
        score_threshold=threshold,
        max_detections=50,
    )
    tmp_metrics = evaluate_map50(base_val_pred_tmp, val_model_df_ignoring_difficult, VOC_CLASSES)
    tmp_map50 = float(tmp_metrics['map50'])

    threshold_rows.append({
        'score_threshold': threshold,
        'val_map50': tmp_map50,
        'val_inference_seconds': val_seconds_tmp,
    })

    if tmp_map50 > best_base_map50:
        best_base_map50 = tmp_map50
        best_base_threshold = threshold

base_threshold_df = pd.DataFrame(threshold_rows)

base_val_pred, base_val_seconds = predict_base_model_to_voc_df(
    base_model,
    val_loader_eval,
    split_name='val',
    score_threshold=best_base_threshold,
    max_detections=50,
)
base_dev_test_pred, base_dev_test_seconds = predict_base_model_to_voc_df(
    base_model,
    dev_test_loader_eval,
    split_name='dev_test',
    score_threshold=best_base_threshold,
    max_detections=50,
)
base_test_pred, base_test_seconds = predict_base_model_to_voc_df(
    base_model,
    test_loader_eval,
    split_name='test',
    score_threshold=best_base_threshold,
    max_detections=50,
)

base_val_metrics = evaluate_map50(base_val_pred, val_model_df_ignoring_difficult, VOC_CLASSES)
base_dev_test_metrics = evaluate_map50(base_dev_test_pred, dev_test_model_df_ignoring_difficult, VOC_CLASSES)
base_test_metrics = evaluate_map50(base_test_pred, test_objects, VOC_CLASSES)

ft_val_metrics = evaluate_map50(best_val_pred, val_model_df_ignoring_difficult, VOC_CLASSES)
ft_dev_test_metrics = evaluate_map50(best_dev_test_pred, dev_test_model_df_ignoring_difficult, VOC_CLASSES)
ft_test_metrics = evaluate_map50(best_test_pred, test_objects, VOC_CLASSES)

base_val_person = evaluate_map50(base_val_pred, val_model_df_ignoring_difficult, ['person'])['map50']
base_dev_test_person = evaluate_map50(base_dev_test_pred, dev_test_model_df_ignoring_difficult, ['person'])['map50']
base_test_person = evaluate_map50(base_test_pred, test_objects, ['person'])['map50']
ft_val_person = evaluate_map50(best_val_pred, val_model_df_ignoring_difficult, ['person'])['map50']
ft_dev_test_person = evaluate_map50(best_dev_test_pred, dev_test_model_df_ignoring_difficult, ['person'])['map50']
ft_test_person = evaluate_map50(best_test_pred, test_objects, ['person'])['map50']

base_dir = EXPERIMENT_RUNS_DIR / 'base_model_zero_shot'
base_dir.mkdir(parents=True, exist_ok=True)

base_val_path = base_dir / 'base_model_val_predictions.csv'
base_dev_test_path = base_dir / 'base_model_dev_test_predictions.csv'
base_test_path = base_dir / 'base_model_test_predictions.csv'
base_tuning_path = base_dir / 'base_model_threshold_tuning.csv'
comparison_path = base_dir / 'base_vs_finetuned_comparison.csv'

base_val_pred.to_csv(base_val_path, index=False)
base_dev_test_pred.to_csv(base_dev_test_path, index=False)
base_test_pred.to_csv(base_test_path, index=False)
base_threshold_df.to_csv(base_tuning_path, index=False)

comparison_df = pd.DataFrame([
    {
        'model': f'best_finetuned::{best_experiment}',
        'threshold': float(best_row['best_threshold']),
        'val_map50_all_classes': float(ft_val_metrics['map50']),
        'dev_test_map50_all_classes': float(ft_dev_test_metrics['map50']),
        'test_map50_all_classes': float(ft_test_metrics['map50']),
        'val_ap50_person': float(ft_val_person),
        'dev_test_ap50_person': float(ft_dev_test_person),
        'test_ap50_person': float(ft_test_person),
        'val_seconds_per_image': float(best_row['val_seconds_per_image']),
        'dev_test_seconds_per_image': float(best_row['dev_test_seconds_per_image']),
        'test_seconds_per_image': float(best_row['test_seconds_per_image']),
    },
    {
        'model': 'base_coco_no_finetune',
        'threshold': float(best_base_threshold),
        'val_map50_all_classes': float(base_val_metrics['map50']),
        'dev_test_map50_all_classes': float(base_dev_test_metrics['map50']),
        'test_map50_all_classes': float(base_test_metrics['map50']),
        'val_ap50_person': float(base_val_person),
        'dev_test_ap50_person': float(base_dev_test_person),
        'test_ap50_person': float(base_test_person),
        'val_seconds_per_image': float(base_val_seconds / max(1, len(val_ds_eval))),
        'dev_test_seconds_per_image': float(base_dev_test_seconds / max(1, len(dev_test_ds_eval))),
        'test_seconds_per_image': float(base_test_seconds / max(1, len(test_ds_eval))),
    },
]).sort_values('val_map50_all_classes', ascending=False).reset_index(drop=True)

comparison_df['delta_vs_base_val_map50'] = comparison_df['val_map50_all_classes'] - float(base_val_metrics['map50'])
comparison_df['delta_vs_base_dev_test_map50'] = comparison_df['dev_test_map50_all_classes'] - float(base_dev_test_metrics['map50'])
comparison_df['delta_vs_base_person_ap50'] = comparison_df['val_ap50_person'] - float(base_val_person)
comparison_df.to_csv(comparison_path, index=False)

print('Best fine-tuned experiment:', best_experiment)
print('Base model threshold (val tuned):', round(best_base_threshold, 2))
print('Saved base val predictions:', base_val_path)
print('Saved base dev_test predictions:', base_dev_test_path)
print('Saved base test predictions:', base_test_path)
print('Saved base threshold tuning:', base_tuning_path)
print('Saved comparison table:', comparison_path)

display(base_threshold_df)
display(comparison_df)

# Optional cleanup for long sessions.
del base_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()